# SPOD to Mapping Excel
- Prerequisites: 
  - Anaconda packages: `xlsxwriter` pandas, openpyxl, seaborn`


## Result

Excel sheet containing:

Sheet with all Datapoints (Systems, Tables and Columns) mapped against the Information Model (Entity, Attribute)
Table covering the overview sheet

Analog dev_x_mapping

Optional:
Sheet per System - IM containing sample data

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet
1. Use row emitter to iterate the whole sheet
    1. Fill cells
    1. Style cells
    1. Protect cells
1. Style whole sheets. Group or hide columns.

## Configuration
The following parameters has to be definded when running as regular python script

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'

DESTINATION = 'SIKA_mapping.xlsx'

MODEL_SOURCE = '/Users/bue/projects/geberit/DEAP/DB/IM_GEBERIT.json'
MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'
MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.regen.json'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [ ]:
# openpyxl
from openpyxl import Workbook
from openpyxl.worksheet.table import Table
from openpyxl.utils.cell import get_column_letter
from openpyxl.styles import PatternFill

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
spod_file = Path(MODEL_SOURCE)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
print(f"Loaded SPOD containing {spod['model']} from '{spod_file.resolve()}'")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    appendix = ''
    if 'columns' == entry:
        mapped_columns_count = len(list(filter(lambda c: len(c['attributesmapped']) > 0, spod[entry].values())))
        appendix = f" (mapped {mapped_columns_count})"
    print(f"- {entry}: {len(spod[entry])}{appendix}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

## List structure definition

In [ ]:
columns_mapped = {}
system_index = {}

In [ ]:
headings_im = [
    "FQN", "EID", "AID", 
    "#", "Level", 
    "Examples", "Description"
]

for lang in spod['languages'].keys():
    headings_im.append(f"Entity ({lang})")
    headings_im.append(f"Attribute ({lang})")

In [ ]:
len(headings_im)

In [ ]:
headings_im

In [ ]:
def sources_first(system: tuple) -> str:
    name = system[1]['name']
    if name == 'SAP-ERP':
        return 'AAA' + name
    if 'SAP' in name:
        return 'ABA' + name
    if 'PIM' in name:
        return 'ACA' + name
    if 'CXM' in name:
        return 'ADA' + name
    return name

In [ ]:
import collections
systems = collections.OrderedDict(sorted(spod['systems'].items(), key=sources_first))
skeys = systems.keys()

In [ ]:
def emit_system_columns(systems: iter) -> [str]:
    result = []
    for key, system in systems.items():
        system_index[key] = len(headings_im) + len(result)
        result.append(system['name'])
    return result

In [ ]:
systems_headings = emit_system_columns(systems)

In [ ]:
logging.info(f"Mapping {len(systems_headings)} systems")

In [ ]:
systems_headings

In [ ]:
def export_column(key: str, column: dict) -> []:
    result = [
                #column['interface-id+'] + '.' + column['table-id'] + '.' + key,
                column['name'],
                #column['interface_col_id'],
            ]
    return result

In [ ]:
headings = headings_im + systems_headings
f"Columns ({len(headings)}): {', '.join(headings)}"

## Append unmapped columns to the bottom

# Prepare output formatting

In [ ]:
import xlsxwriter

In [ ]:
xlsx_destination = Path(DESTINATION)
workbook = xlsxwriter.Workbook(xlsx_destination)

In [ ]:
title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})
column_head_format = workbook.add_format({'bold': True, 'bg_color': '#A0A0A0'})

column_head_format = workbook.add_format({'bg_color': '#A0A0A0'})

In [ ]:
title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})
column_head_format = workbook.add_format({'bold': True, 'bg_color': '#A0A0A0'})

level_colors = [
    workbook.add_format({'bg_color': '#FAEC2D'}),
    workbook.add_format({'bg_color': '#FAAC0D'}),
    workbook.add_format({'bg_color': '#AB8D6D'}),
    workbook.add_format({'bg_color': '#AB0000'}),
    workbook.add_format({'bg_color': '#008D00'}),
]

unleveled = len(level_colors) + 1
unleveled

In [ ]:
## Summary is first sheet, but will be filled last
summary = workbook.add_worksheet('Summary')

In [ ]:
## Create 'Mapping' sheet

In [ ]:
worksheet = workbook.add_worksheet('Mapping')

### Sort by Attribute FQN
#data_table.sort(key=lambda r: r[0] if r[0] is not None else '\uFFFF')


### Headers

In [ ]:
col = 0
for header in headings:
    worksheet.write(0, col, header, title_format)
    col += 1

### Mapped columns

In [ ]:
def get_sika_level(column: dict) -> int:
    mappings = column['userdefprops'].get('column-mapping', {}).get('EXTERNAL', {})
    for mapping in mappings.values():
        value = mapping.get('value')
        #print(f"{mapping} = {value}")
        if mapping.get('name') == 'SIKA-PACKAGING-LEVEL' and value is not None:
            try:
                return int(value)
            except ValueError:
                pass
    return unleveled

In [ ]:
get_sika_level(next(iter(map(lambda t: t[1], filter(lambda t: 'TYPE_SURCOND' in t[1]['name'], spod['columns'].items())))))

In [ ]:
leveled = list(filter(lambda c: get_sika_level(c) != unleveled, spod['columns'].values()))
len(leveled)

In [ ]:
leveled_without_mapping = list(filter(lambda c: len(c['attributesmapped']) < 1, leveled))
len(leveled_without_mapping)

In [ ]:
def is_mapped_on_level(spod: dict, column_key: str, attribute_key: str, level: int) -> bool:
    column = spod['columns'][column_key]
    if attribute_key in column['attributesmapped']:
        if level is not None:
            return level == get_sika_level(column)
        else:
            return get_sika_level(column) == unleveled

In [ ]:
logging.info(f"Sika annotated columns: {len(leveled_without_mapping)}/{len(leveled)}")

In [ ]:
def firsthit(spod: dict, attribute_key: str, system_key: str, level: int) -> []:    
    for ckey, column in spod['columns'].items():
        if ckey not in columns_mapped and system_key == column['interface-id+'] and attribute_key in column['attributesmapped']:
            if level == None:
                return (ckey, column)
            if level == get_sika_level(column):
                return (ckey, column)
    return (None, None)

In [ ]:
def write_columns(spod: dict, worksheet, row, col, columns, level) -> int:
    
    if level == None or level == unleveled:
        cell_format = None
    else:
        cell_format = level_colors[level]
        
    items = 0
    for skey, system in systems.items():
        for ckey in columns:
            if columns_mapped.get(ckey) is None:
                column = spod['columns'][ckey]
                if column['interface-id+'] == skey:
    #        key, column = firsthit(spod, attribute_key, skey, level)
    #        if key is not None:
                # mark column as processed
                    columns_mapped[ckey] = column
                    worksheet.write(row, col, column['name'], cell_format)
                    items += 1
        col += 1
        
    return items

In [ ]:
# Print headings to look up in write_row
index = 0
for title in headings_im:
    print(f"{index:02} {title}")
    index += 1

In [ ]:
systems_start_column = index
systems_start_column

In [ ]:
def write_row(worksheet, row, spod: dict, attribute_key: str, attribute: dict, level: int, translator: Translator) -> []:
    enti_key = attribute['entity']
    entity = spod['entities'].get(enti_key)
    assert entity is not None, f"Missing entity {enti_key}"
    if level is not None and level < len(level_colors):
        style = level_colors[level]
    else:
        style = None
        
    col = 0
    worksheet.write(row, col, enti_key + ':' + attribute_key, style)
    worksheet.write(row, col + 1, enti_key, style)
    worksheet.write(row, col + 2, attribute_key, style)
    
    columns_on_level = list(filter(lambda ckey: is_mapped_on_level(spod, ckey, attribute_key, level), spod['columns'].keys()))
    worksheet.write(row, col + 3, len(columns_on_level), style)
    worksheet.write(row, col + 4, level, style)
    
    col = 5
    worksheet.write(row, col, ', '.join(translator.tr(attribute.get('examples'))), style)
    worksheet.write(row, col + 1, translator.tr(attribute['descr']), style)
       
    col = 7
    for lang in spod['languages'].keys():
        worksheet.write(row, col, translator.tr(entity['name'], lang))
        worksheet.write(row, col + 1, translator.tr(attribute['name'], lang))
        col += 2
        
    result = [
        enti_key + ':' + attribute_key,
        enti_key,
        attribute_key,
        translator.tr(entity['name'], 'en'),
        translator.tr(attribute['name'], 'en'),
        translator.tr(entity['name'], 'de'),
        translator.tr(attribute['name'], 'de'),
        translator.tr(entity['name'], 'fr'),
        translator.tr(attribute['name'], 'fr'),
        ', '.join(translator.tr(attribute.get('examples'), 'en')),
        translator.tr(attribute['descr'], 'en'),
        len(attribute['columnsmapped+']),
    ]
    assert col == systems_start_column, f"Column index {col} != {systems_start}"
    col = systems_start_column
    written = write_columns(spod, worksheet, row, col, columns_on_level, level)
    col += len(skeys)
    
    #print(f"Wrote {written} on row {row}")
    # Separator column
    #col += 1
    #worksheet.write(row, col, column['name'], cell_format)
        

    return written

In [ ]:
mapped = 0
row = 1

unmapped = set()

# Iterate levels first
for packaging_level in range(0, len(level_colors) + 1):
    lvl = None if packaging_level >= len(level_colors) else packaging_level + 1
    ccount = 0
    for key, attribute in spod['attributes'].items():
        columns_on_level = list(filter(lambda ckey: is_mapped_on_level(spod, ckey, key, packaging_level), spod['columns'].keys()))
        if len(columns_on_level) > 0:
            #logger.info(f"Columns on level {lvl} for {key}: {columns_on_level}")
            entry = write_row(worksheet, row, spod, key, attribute, packaging_level, translator)
            mapped += 1 if len(columns_on_level) > 0 else 0
            row += 1        
            ccount += len(columns_on_level)
        else:
            # Unmapped attribute -> later
            unmapped.add(key)
    logger.info(f"Collected {ccount} for level {lvl}")
    
logger.info(f"Wrote {row-1} attribute rows. {mapped} are mapped to columns. Unmapped attributes remaining: {len(unmapped)}")

### Unmapped attributes

In [ ]:
columns_written = 0

rstart = row
for key in unmapped:
    attribute = spod['attributes'][key]
#    columns_on_level = set(filter(lambda ckey: is_mapped_on_level(spod, ckey, key, unleveled), spod['columns'].keys()))
#    columns_unmapped = columns_on_level - set(columns_mapped.keys())
#    if len(columns_mapped) > 0:
        #assert len(columns_on_level) == 0
    columns_on_line = write_row(worksheet, row, spod, key, attribute, unleveled, translator)
    columns_written += columns_on_line
    row += 1        

logger.info(f"Wrote {columns_written} columns on {row - rstart} from unmapped {len(unmapped)}")

### Unmapped columns

In [ ]:
def write_aux_row(worksheet, row: int, spod: dict, key: str, column: dict, translator: Translator) -> []:
    mapped = column['attributesmapped']
    if len(mapped) > 0:
        attrkey = mapped[0]
        attr = spod['attributes'][attrkey]
        #assert False, f"Column {key} is mapped to {mapped} {translator.tr(attr['name'])}"
        result = emit_row(worksheet, row, spod, attrkey, attr, translator)
        result.extend( [ None ] * (len(headings) - len(result))  )
    else:
        result = [ None ] * len(headings)
    
    map_count = len(column['attributesmapped'])
    result[len(headings_im) - 1] = map_count

    index = system_index[column['interface-id+']]
    values = export_column(key, column)
    result[index + 0] = values[0]

    column = index
    for value in values:
        worksheet.write(row, column, value) 
    
    return result

In [ ]:
remainder = set(spod['columns'].keys()) - set(columns_mapped.keys())
logging.info(f"Unprocessed columns remaining {len(remainder)}")

In [ ]:
#remainder = list(filter(lambda key: key not in columns_mapped.keys(), spod['columns'].keys()))

for ckey in tqdm(remainder, desc="Unmapped columns", dynamic_ncols=True):
    column = spod['columns'][ckey]
    #entry = write_aux_row(worksheet, row, spod, ckey, column, translator)
    level = get_sika_level(column)
    written = write_columns(spod, worksheet, row, systems_start_column, [ckey], level)
    
    row += 1

### Define Table

In [ ]:
table_column_headers = [ { 'header': name } for name in headings ]

In [ ]:
worksheet.add_table(0, 0, row + 1, len(headings) - 1, { 
    'name': 'mapping',
    'banded_rows': True,
    'columns': table_column_headers,
})

### Styling

In [ ]:
# FQN width
worksheet.set_column(0, 0, 20)

# Hide EID, AID on the left
worksheet.set_column(1, 3, 10, None, { 'hidden': 1, })

# EID, AID
worksheet.set_column(3, 5, 20)

# Hide attribute name translations (DE, FR)
worksheet.set_column(5, 8, 40, None, { 'hidden': 1, })

base = len(headings_im)
index = 0
for system in skeys:
#    colnr = base + (index * 3)
    colnr = base + index
#    worksheet.set_column(colnr, colnr, None, None, { 'hidden': 1, })
    
    # Column name on system
    worksheet.set_column(colnr + 1, colnr + 1, 30)
    
    # Technical reference
 #   worksheet.set_column(colnr + 2, colnr + 2, None, None, { 'hidden': 1, })        
    index += 1

## Add one sheet per system

In [ ]:
def fill_worksheet(spod: dict, skey: str, system: str, sheet):
    row = 0
    sheet.write(row, 0, 'Table Key', column_head_format)
    sheet.write(row, 1, 'Table Name', column_head_format)
    sheet.write(row, 2, 'Column Key', column_head_format)
    sheet.write(row, 3, 'Tech-ID', column_head_format)
    sheet.write(row, 4, 'Name', column_head_format)
    sheet.write(row, 5, 'Description', column_head_format)
    sheet.write(row, 6, '|', column_head_format)
    sheet.write(row, 7, 'IM Attributes', column_head_format)
    
    row += 1
    columns = sorted(list(spod['columns'].items()), key=lambda c: c[1]['table-name+'])
    for ckey, column in columns:
        if column['interface-id+'] == skey:
            sheet.write(row, 0, column['table-id'])
            sheet.write(row, 1, column['table-name+'])
            sheet.write(row, 2, ckey)
            sheet.write(row, 3, column['interface_col_id'])
            sheet.write(row, 4, column['name'])
            sheet.write(row, 5, translator.tr(column['descr'], 'de'))
            sheet.write(row, 6, '|')
            sheet.write(row, 7, ', '.join(column['attributesmapped']))
            row += 1
    
    return row

In [ ]:
import re

worksheets = dict()

for key, system in systems.items():
    title = re.sub(r'[\:\[\]*?/\\]', '_', system['name'])
    length = min(25, len(title))
    t = key.replace('INTF','') + ' ' + title[:length]
    if t.lower() in worksheets.keys():
        t = key.replace('INTF','') + ' ' + title[max(0, len(title) - 25):]
    worksheet = workbook.add_worksheet(t)
    worksheets[t.lower()] = worksheet
    rows = fill_worksheet(spod, key, system, worksheet)
    print(f"{key}: {system['name']} -> {t} with {rows} rows")

## Summary sheet

In [ ]:
summary.write(0, 0, "Summary", title_format)

row = 2
summary.write(row, 0, 'Key', column_head_format)
summary.write(row, 1, 'Name', column_head_format)
summary.write(row, 2, 'Mapped', column_head_format)
summary.write(row, 3, 'Total', column_head_format)
summary.write(row, 4, 'Tables', column_head_format)
summary.write(row, 5, 'Sheet Link', column_head_format)

row = 3
for skey, system in systems.items():
    summary.write(row, 0, skey)
    summary.write(row, 1, system['name'])
    
    columns = list(filter(lambda c: c['interface-id+'] == skey, spod['columns'].values()))
    mapped = list(filter(lambda c: len(c['attributesmapped']) > 0, columns))
    summary.write(row, 2, len(mapped))
    summary.write(row, 3, len(columns))
    
    summary.write(row, 4, len(system['tables+']))
    
    _, ws = next(iter(filter(lambda t: skey[4:] in t[0], worksheets.items())))
    summary.write_url(row, 5, f"internal:'{ws.get_name()}'!A1", string=f'Sheet {skey[4:]}')
    
    row += 1

## Styling

In [ ]:
summary.set_column(0, 0, 20)
summary.set_column(1, 1, 60)
summary.set_column(2, 5, 15)

## Write Excel file

In [ ]:
version_file = Path(LIBRARY, 'versons.json')
if version_file.is_file():
    with open(version_file, 'r') as src:
        version = json.load(src)
else:
    version = { 'TOOLVERSION': '?.?' }

In [ ]:
workbook.set_properties({
    'title':    f"{spod['model']['name']}",
    'subject':  'mapping',
    'author':   f"Excel Mapping Publisher {version['TOOLVERSION']}",
#    'manager':  'D',
#    'company':  'of Wolves',
    'category': 'export',
    'keywords': 'Information Model, Data Models',
    'comments': f"generated with {version['TOOLVERSION']} from model {spod['_imprint_']['Modelversion']}",
    'status':   'Draft',
    'revision': spod['_imprint_'].get('git')
})

In [ ]:
workbook.close()
print(f"Wrote {xlsx_destination}")

# Visually verify

In [ ]:
import subprocess
r = subprocess.run(['qlmanage', '-x', '-p', xlsx_destination], shell=False) # capture_output=False, stderr=subprocess.DEVNULL)

In [ ]:
import pandas
excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [ ]:
from IPython.display import display, HTML
display(excel_data_df)